In [1]:
import pandas as pd
import os

Data Extraction

In this first step, we are going to extract the raw data from the CSV files located in the 4datasets folder. We will use the pandas library to create a DataFrame, which is a structured table that allows us to manipulate and analyze the data efficiently. Our goal is to load the olist_orders_dataset.csv and inspect its structure to ensure the data is formatted correctly before starting the ETL process.

In [2]:
file_path = '4datasets/olist_orders_dataset.csv' #path to the dataset

df_orders = pd.read_csv(file_path) #Load the dataset into a pandas DataFrame

print("--- Dataset Overview ---") 
print(df_orders.info())  #summary info
df_orders.head()  #first few rows and the

--- Dataset Overview ---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


From the output of df_orders.info(), date columns (such as order_purchase_timestamp) are read as str (strings/text). To perform temporal analysis or calculate shipping times, we need to convert them to datetime format. Additionally, some delivery date values ​​are missing (null values).

Data Transformation and Cleaning

In this stage, we refine our dataset to make it suitable for analysis. The most critical task is converting date-related columns from generic text strings into proper datetime objects. This allows us to perform time-series analysis and calculate intervals between order placement and delivery. We will also identify missing values within the delivery columns, which is essential for understanding logistics performance and handling incomplete order cycles.

In [3]:
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
] #extract only the columns that should be treated as dates

for col in date_columns:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce') #convert strings to datetime objects

In [4]:
missing_data = df_orders.isnull().sum() #check for missing values in each column

print("--- Missing Values Count ---")
print(missing_data)

--- Missing Values Count ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


2965 null values ​​in order_delivered_customer_date. This means that your dataset contains nearly three thousand orders that have been shipped or processed but have not yet arrived.

In [5]:
# Show the new data types to confirm the change
print("\n--- Updated Column Types ---")
print(df_orders.dtypes)


--- Updated Column Types ---
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


Logistics Performance Analysis

Now that our date columns are properly formatted as datetime objects, we can calculate the actual delivery time for each order. This is done by subtracting the purchase timestamp from the customer delivery date. We will focus on the "delivered" orders to find the average fulfillment duration. Additionally, we will identify potential outliers—orders that took an unusually long time to arrive—which is a key metric for evaluating courier performance and customer satisfaction.

In [6]:
# Calculate delivery time in days
# We subtract the purchase date from the delivery date and extract the number of days
df_orders['delivery_time_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.days
#dt.days: it will transfrom the results from hr/min/sec in int (number of days)

# Calculate the average delivery time
# Filter only delivered orders to get a clean average
delivered_orders = df_orders[df_orders['order_status'] == 'delivered']
avg_delivery = delivered_orders['delivery_time_days'].mean() #mean calculates the avarage ignoring the null values

In [7]:
print(f"--- Logistics Metrics ---")
print(f"Average Delivery Time: {avg_delivery:.2f} days")

# 5. Identify and display the top 5 longest deliveries (Potential Outliers)
print("\n--- Top 5 Longest Deliveries (Outliers) ---")
longest_deliveries = delivered_orders[['order_id', 'delivery_time_days']].sort_values(by='delivery_time_days', ascending=False)
#.sort_values(...): order from slowest to fastest  
print(longest_deliveries.head(5))

--- Logistics Metrics ---
Average Delivery Time: 12.09 days

--- Top 5 Longest Deliveries (Outliers) ---
                               order_id  delivery_time_days
19590  ca07593549f1816d26a572e06dc1eab6               209.0
55619  1b3190b2dfa9d789e1f14c05b647a14a               208.0
61610  440d0d17af552815d15a9e41abe49359               195.0
70307  2fb597c2f772eca01b1f5c561bf6cc7b               194.0
89130  285ab9426d6982034523a855f55a885e               194.0


The analysis reveals an average delivery time of 12.09 days, which serves as the primary performance benchmark for the Olist platform. However, the detection of extreme outliers is the most significant finding in this step:

Maximum Delay: We identified orders taking up to 209 days to reach the customer.

Operational Impact: Such extreme values (190+ days) indicate severe logistical failures, potentially due to lost shipments, incorrect data entry, or regional infrastructure issues.

Next Objective: To add business value, we must now determine if these delays are widespread or concentrated in specific Brazilian states. This will allow us to separate standard performance from localized logistical bottlenecks.

Merging Datasets
To gain a deeper understanding of the logistics bottlenecks identified in the previous step, we need to correlate delivery times with geographical data. Currently, the orders dataset only contains timestamps and status information, but lacks the customer's location.

In this step, we will:

-Extract the customer location data from     olist_customers_dataset.csv.

-Transform the data by performing a Merge (Join) operation between our filtered delivered_orders and the customers dataset, using the customer_id as the common key.

-Analyze the average delivery performance grouped by state (customer_state) to identify which regions of Brazil experience the most significant delays.

In [8]:
#Load the customers dataset
customers_path = '4datasets/olist_customers_dataset.csv'
df_customers = pd.read_csv(customers_path)

#Merge orders with customers to get the state information
# This combines the delivery time data with the geographical location
df_geo_logistics = pd.merge(delivered_orders, df_customers, on='customer_id')

#Calculate average delivery time per state
# We group by state and calculate the mean of the delivery days
state_performance = df_geo_logistics.groupby('customer_state')['delivery_time_days'].mean().sort_values(ascending=False)

print("--- Average Delivery Time by State (Top 10 Slowest) ---")
print(state_performance.head(10))
#in summary we take the data, group it by state, calculate the average of the delivery days for each group, and show the worst (slowest) ones first

--- Average Delivery Time by State (Top 10 Slowest) ---
customer_state
RR    28.975610
AP    26.731343
AM    25.986207
AL    24.040302
PA    23.316068
MA    21.117155
SE    21.029851
CE    20.817826
AC    20.637500
PB    19.953578
Name: delivery_time_days, dtype: float64


The output clearly identifies a significant geographical gap in logistics performance. While the national average is approximately 12 days, states like RR (Roraima) and AP (Amapá) suffer from average wait times exceeding 28 days.

This disparity suggests that infrastructure challenges in the North and Northeast regions of Brazil are the primary drivers of extreme delivery outliers.

Customers in these regions have a vastly different experience compared to those in the South, which may affect churn rates and satisfaction scores in those specific areas.

Data Consolidation and Export (Load)

We have completed the Extract and Transform phases. Now, we move to the final part of the ETL process: Load. We want to save our enriched dataset (the one with delivery days and customer states) into a new file. This allows us or other team members to use the cleaned data later without running the whole script again.

Consolidate: Filter the final merged table to keep only the columns we need.

Export: Save the result to a new CSV file named cleaned_olist_delivery_data.csv.

In [9]:
#Select relevant columns for the final cleaned dataset
final_columns = [
    'order_id', 'customer_id', 'customer_state', 'customer_city',
    'order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_time_days'
]
df_final = df_geo_logistics[final_columns] #create a dataframe with only what we need

#Save the cleaned data to a new CSV file
output_path = 'cleaned_olist_delivery_data.csv' #we define the name of file
df_final.to_csv(output_path, index=False) #index=False otherwise Pandas would add extra columns at the beginning with row numbers 0,1,
print(f"--- ETL Process Complete ---")
print(f"Cleaned dataset saved as: {output_path}")
print(f"Final file contains {df_final.shape[0]} rows.")

--- ETL Process Complete ---
Cleaned dataset saved as: cleaned_olist_delivery_data.csv
Final file contains 96478 rows.
